In [ ]:
import boto3
import json
import time
import threading
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

REGION = "us-east-1"
ACCOUNT = "857753985214"

sm = boto3.client("sagemaker", region_name=REGION)
ssm_client = boto3.client("ssm", region_name=REGION)
lam = boto3.client("lambda", region_name=REGION)

PROFILE_TO_USER = {
    "Vasif-test-user": "Vasif_test_user",
    "Aghamir-test-user": "Aghamir_test_user",
    "Murad-test-user": "Murad_test_user",
}

# --- detect current user ---
try:
    arn = boto3.client("sts", region_name=REGION).get_caller_identity()["Arn"]
    current_user = next(
        (u for p, u in PROFILE_TO_USER.items() if p.lower().replace("-","") in arn.lower().replace("-","")),
        "unknown"
    )
except Exception:
    current_user = "unknown"

# --- list models ---
canvas_models = []
try:
    jobs = sm.list_auto_ml_jobs(
        StatusEquals="Completed",
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=20
    )["AutoMLJobSummaries"]
    for job in jobs:
        try:
            tags = {t["Key"]: t["Value"] for t in sm.list_tags(
                ResourceArn=f"arn:aws:sagemaker:{REGION}:{ACCOUNT}:automl-job/{job['AutoMLJobName']}"
            ).get("Tags", [])}
            profile_name = tags.get("sagemaker:user-profile-arn", "").split("/")[-1]
            job_user = PROFILE_TO_USER.get(profile_name, "unknown")
            if job_user != current_user and current_user != "unknown":
                continue
            model_name = tags.get("CanvasModelName", "")
            label = f"{model_name} — {job['CreationTime'].strftime('%Y-%m-%d %H:%M')}"
            canvas_models.append({"label": label, "job": job["AutoMLJobName"], "model": model_name})
        except Exception:
            pass
except Exception as e:
    print(f"Error: {e}")

# --- UI ---
title = widgets.HTML(value="<h2>Run Full Training Pipeline</h2><p>Select a Canvas model and click Run.</p>")

user_label = widgets.HTML(value=f"<b>User:</b> {current_user}")

dropdown = widgets.Select(
    options=[(m["label"], i) for i, m in enumerate(canvas_models)] if canvas_models else [("No Canvas models found", -1)],
    rows=min(5, max(1, len(canvas_models))),
    layout=widgets.Layout(width="700px")
)

run_btn = widgets.Button(
    description="Run Full Pipeline",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="200px", height="45px"),
    disabled=(not canvas_models)
)

refresh_btn = widgets.Button(
    description="Refresh",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(width="120px", height="45px")
)

out = widgets.Output()
state = {"job": None, "model": None}

def get_bucket(model_name):
    if not model_name:
        return "canvas-datasets-vorujzada"
    n = model_name[6:] if model_name.startswith("Model_") else model_name
    clean = "".join(c if c.isalnum() or c=="-" else "-" for c in n.lower())
    return f"azc-ml-prj-{clean}"

def refresh_status():
    with out:
        clear_output(wait=True)
        if not state["job"]:
            print("No pipeline started yet.")
            return
        job = state["job"]
        bucket = get_bucket(state["model"])
        lines = [f"<b>Canvas job:</b> {job}",
                 f"<b>Project bucket:</b> s3://{bucket}/"]
        try:
            ssm_client.get_parameter(Name=f"/canvas/retrain-triggered/{job}")
            lines.append("&#x2705; Pipeline triggered")
        except Exception:
            lines.append("&#x23f3; Waiting for pipeline (up to 5 min)...")
        try:
            tj = sm.list_training_jobs(SortBy="CreationTime", SortOrder="Descending", MaxResults=5)["TrainingJobSummaries"]
            rt = [j for j in tj if "retrain-xgb" in j["TrainingJobName"]]
            if rt:
                st = rt[0]["TrainingJobStatus"]
                icon = {"Completed":"&#x2705;","InProgress":"&#x23f3;","Failed":"&#x274c;"}.get(st,"?")
                lines.append(f"{icon} Training: {rt[0]['TrainingJobName']} — {st}")
                if st == "Completed":
                    lines.append(f"&#x1f389; Model: s3://{bucket}/model/candidate/")
        except Exception:
            pass
        try:
            bj = sm.list_transform_jobs(SortBy="CreationTime", SortOrder="Descending", MaxResults=3)["TransformJobSummaries"]
            rb = [j for j in bj if "batch-" in j["TransformJobName"]]
            if rb:
                st = rb[0]["TransformJobStatus"]
                icon = {"Completed":"&#x2705;","InProgress":"&#x23f3;","Failed":"&#x274c;"}.get(st,"?")
                lines.append(f"{icon} Batch: {rb[0]['TransformJobName']} — {st}")
        except Exception:
            pass
        display(HTML("<br>".join(lines)))

def on_run(b):
    if dropdown.value is None or dropdown.value == -1:
        return
    m = canvas_models[dropdown.value]
    state["job"] = m["job"]
    state["model"] = m["model"]
    try:
        ssm_client.delete_parameter(Name=f"/canvas/retrain-triggered/{m['job']}")
    except Exception:
        pass
    run_btn.disabled = True
    run_btn.description = "Starting..."
    try:
        lam.invoke(
            FunctionName="CanvasAutoRetrain",
            InvocationType="Event",
            Payload=json.dumps({"automl_job_name": m["job"], "user": current_user, "canvas_model_name": m["model"]}).encode()
        )
        run_btn.description = "Run Full Pipeline"
        run_btn.disabled = False
        refresh_status()
        def poll():
            for _ in range(40):
                time.sleep(30)
                refresh_status()
        threading.Thread(target=poll, daemon=True).start()
    except Exception as e:
        run_btn.description = "Run Full Pipeline"
        run_btn.disabled = False
        with out:
            clear_output()
            print(f"Error invoking Lambda: {e}")

run_btn.on_click(on_run)
refresh_btn.on_click(lambda b: refresh_status())

display(widgets.VBox([
    title,
    user_label,
    widgets.HTML("<b>Select Canvas model:</b>"),
    dropdown,
    widgets.HBox([run_btn, refresh_btn]),
    out
]))